In [ ]:
#| default_exp sess

# sess

> Find and read a session from either host

In [ ]:
from fastcore.test import *
from importlib.resources import files
from aidialog.dialog import Dialog
from aidialog.dlgskill import summary_dlg
from aidialog.ipynb import read_ipynb
import json, shutil, tempfile

In [ ]:
#| export
from fastcore.script import call_parse, is_cli
from fastcore.utils import *
import re
from aidialog.msg_parts import Msg, Text, ToolUse
from aidialog.hist import chat2dlg
from aidialog.ipynb import write_ipynb
from llmsurgery import ant, oai

Claude Code sessions and Codex threads are stored differently and read by different modules, but a person reaching for one has only an id and a question. `find_sess` takes that id, from either host, and says which host owns it and where the transcript is; `sess_dlg` reads it into a dialog for the aidialog tools. Ids may be given as any unique prefix.

Where `ant.sess2dlg` and `oai.thread2dlg` are faithful conversions, `sess_dlg` is a reading view: it drops the host's bookkeeping and the harness's injected turns, and it reaches back through compactions by default, so the dialog holds the conversation rather than the machinery around it. The `sess2nb` command line writes that view to an ipynb.

## Finding a session

In [ ]:
#| export
def find_sess(
    ref=None, # Session id or unique id prefix; the current session if None
    cwd=None, # Project directory, for Claude sessions
    codex_home=None, # Codex home; `oai.CODEX_HOME` if None
):
    "The host owning session `ref` and its transcript path: `('ant'|'oai', path)`"
    if ref is None: ref = oai.cur_thread() or ant.cur_sess(cwd)
    paths = dict(ant=ant.sess_file(ref,cwd), oai=oai.rollout_file(ref, codex_home or oai.CODEX_HOME))
    found = {h:p for h,p in paths.items() if p and p.exists()}
    if not found: raise FileNotFoundError(f'No Claude session or Codex thread for {ref!r}')
    if len(found)>1: raise ValueError(f'{ref!r} names both a Claude session and a Codex thread')
    return first(found.items())

The fixtures the `ant` and `oai` notebooks use are ordinary transcripts, so putting one where each host keeps its sessions is enough to find it. A Claude transcript is named by its session id, and eight characters of that id are plenty:

In [ ]:
proj = Path(tempfile.mkdtemp())
sd = ant.sess_dir(proj)
sd.mkdir(parents=True)
shutil.copy(Path(files('llmsurgery')/'data'/'ant'/'source.jsonl'), sd/'ab12cd34-0000-4000-8000-000000000000.jsonl')
find_sess('ab12cd34', proj)

A Codex rollout is found the same way, from a thread id embedded in a longer filename, so `find_sess` also reports which host owns the id it was given:

In [ ]:
home = Path(tempfile.mkdtemp())
ses = home/'sessions'/'2026'/'01'
ses.mkdir(parents=True)
shutil.copy(Path(files('llmsurgery')/'data'/'oai'/'source.jsonl'), ses/'rollout-2026-01-24T02-44-30-ef56ab78.jsonl')
find_sess('ef56ab78', codex_home=home)

`recent_sess` answers "the session I just finished here" without needing any id: the newest transcript for a directory on either host -- the project's transcript folder on the Claude side, `project_thread` on the Codex side -- whichever was written to last:

In [ ]:
#| export
def recent_sess(
    cwd=None, # Project directory; the current directory if None
    codex_home=None, # Codex home; `oai.CODEX_HOME` if None
):
    "The host with the newest transcript for `cwd`, and its path: `('ant'|'oai', path)`"
    cwd = cwd or '.'
    found = {}
    ap = max(ant.sess_dir(cwd).glob('*.jsonl'), key=os.path.getmtime, default=None)
    if ap: found['ant'] = ap
    try: found['oai'] = oai.project_thread(cwd, codex_home or oai.CODEX_HOME)[1]
    except FileNotFoundError: pass
    if not found: raise FileNotFoundError(f'No Claude session or Codex thread for {Path(cwd).resolve()}')
    return max(found.items(), key=lambda it: it[1].stat().st_mtime)

In [ ]:
rp = ses/'rollout-2026-01-25T09-00-00-aabbccdd.jsonl'
rp.write_text(json.dumps(dict(type='session_meta',payload=dict(id='aabbccdd',cwd=str(proj))))+'\n')
ap = sd/'ab12cd34-0000-4000-8000-000000000000.jsonl'
mt = ap.stat().st_mtime+1
os.utime(rp,(mt,mt))
test_eq(recent_sess(proj,home), ('oai',rp))
os.utime(ap,(mt+1,mt+1))
test_eq(recent_sess(proj,home), ('ant',ap))
recent_sess(proj,home)

## Reading a session

Both hosts record a transcript in append order, so reading the whole history needs no chain walking: for Claude, `ant.conv_recs` keeps the conversation records in the order they happened, and for Codex, `oai.response_items` returns every recorded item, superseded history included. `since_compact` swaps each for its narrower counterpart, `ant.sess_thread` and `oai.active_items`, giving only what the model would see now.

Filtering is what makes this a reading view rather than a transcript. Compaction summaries go, because with the full history present they restate what is already there. Turns the harness injected go too: a skill body arrives as a user turn indistinguishable from a typed one except by its opening line. What remains is what a person said and what the assistant said back.

In [ ]:
#| export
injected_starts = (
    'Base directory for this skill:', 'Stop hook feedback:', '[Request interrupted by user]',
    '<system-reminder>', '<local-command-stdout>', '<command-name>', '<environment_context>',
    '# AGENTS.md instructions for')

def _msg_txt(m): return ''.join(p.text or '' for p in m.content if isinstance(p, Text))

def _injected(m):
    if m.role!='user': return False
    txt = _msg_txt(m).strip()
    return txt.startswith(injected_starts) or bool(re.fullmatch(r'/[\w-]+(\s+\S+)?', txt))

def _no_tools(msgs):
    res = []
    for m in msgs:
        if m.role=='tool': continue
        if ps := [p for p in m.content if not isinstance(p, ToolUse)]: res.append(Msg(role=m.role, content=ps))
    return res

In [ ]:
#| export
def sess_chat(
    host, # `'ant'` for a Claude session, `'oai'` for a Codex thread
    path, # The transcript path, e.g. from `find_sess`
    since_compact=False, # Only the conversation since the last compaction?
):
    "Canonical messages for the conversation recorded in `path`, oldest first"
    if host=='ant':
        recs = ant.load_recs(path)
        recs = ant.conv_recs(ant.sess_thread(recs) if since_compact else recs)
        seen,uniq = set(),[]
        for r in recs:
            if r['uuid'] not in seen:
                seen.add(r['uuid'])
                uniq.append(r)
        recs = uniq
        return ant.recs2chat([r for r in recs if not (r.get('isCompactSummary') or r.get('llmsurgeryCompact'))])
    recs = oai.load_recs(path)
    return oai.items2chat(oai.conv_items(oai.active_items(recs) if since_compact else oai.response_items(recs)))

In [ ]:
#| export
def path_dlg(
    host, # `'ant'` for a Claude session, `'oai'` for a Codex thread
    path, # The transcript path, e.g. from `find_sess`
    name=None, # Dialog name; the transcript's stem if None
    mx=10, # Maximum characters per rendered tool input/output string; None disables truncation
    since_compact=False, # Only the conversation since the last compaction?
    strip_tools=False, # Drop tool calls and their results entirely?
):
    "The conversation of the transcript at `path` as a dialog, its nb meta recording source and time span"
    msgs = [m for m in sess_chat(host, path, since_compact) if not _injected(m)]
    if strip_tools: msgs = _no_tools(msgs)
    dlg = chat2dlg(msgs, name or Path(path).stem, mx=mx)
    times = sorted(t for m in msgs if (t := (getattr(m, 'meta', None) or {}).get('created')))
    span = dict(created=times[0], last=times[-1]) if times else {}
    dlg.meta = dict(llmsurgery=dict(host=host, source=str(path), **span))
    return dlg

def sess_dlg(
    ref=None, # Session id or unique id prefix; the current session if None
    cwd=None, # Project directory, for Claude sessions
    codex_home=None, # Codex home; `oai.CODEX_HOME` if None
    name=None, # Dialog name; the transcript's id if None
    mx=10, # Maximum characters per rendered tool input/output string; None disables truncation
    since_compact=False, # Only the conversation since the last compaction?
    strip_tools=False, # Drop tool calls and their results entirely?
):
    "The conversation of a Claude session or Codex thread as a dialog, ready to read or save"
    host,path = find_sess(ref, cwd, codex_home)
    return path_dlg(host, path, name or str(ref or path.stem), mx=mx, since_compact=since_compact, strip_tools=strip_tools)

In [ ]:
sess_dlg('ab12cd34', proj).summary()

The dialog is self-describing: nb-level meta records which transcript it came from and the conversation's true time span (first and last record timestamps), and each message's cell meta carries its source record's `created` time and `uid`. File times can lie -- opening an old session bumps its mtime -- so provenance lives in the data:

In [ ]:
md = sess_dlg('ab12cd34', proj)
lm = md.meta['llmsurgery']
test_eq(lm['host'], 'ant')
assert lm['source'].endswith('.jsonl')
assert lm['created'] <= lm['last']
assert all(m.meta['created'] and m.meta['uid'] for m in md.messages)
lm

Claude transcripts can contain the same record twice (chain restarts replay records into the append-order file). A replayed record is the same message, so the reading view keeps the first occurrence -- duplicating every line of a transcript changes nothing:

In [ ]:
ap = sd/'ab12cd34-0000-4000-8000-000000000000.jsonl'
lines = ap.read_text().splitlines()
ap.write_text('\n'.join(lines+lines)+'\n')
d2 = sess_dlg('ab12cd34', proj)
test_eq([m.id for m in d2.messages], [m.id for m in md.messages])
test_eq([m.content for m in d2.messages], [m.content for m in md.messages])

`strip_tools` leaves only what was said, which is the form to read when the question is what was decided rather than what was run:

In [ ]:
bare = sess_dlg('ab12cd34', proj, strip_tools=True)
assert not any('{.tool}' in (m.ai_res or '') for m in bare.messages)
bare.summary()

## The command line

`sess2nb` writes that dialog to an ipynb, so a session becomes a notebook to read, search, or paste from. It returns the path when called from Python and prints it when run as a command. `-r` converts the newest session for the current directory, from either host, so no id is needed for the session just finished.

In [ ]:
#| export
@call_parse(pos=['ref'])
def sess2nb(
    ref:str=None, # Session id or unique id prefix
    Out:str=None, # Output path, `.ipynb` added if missing; `<id>.ipynb` in the current directory if None
    mx:int=10, # Maximum characters per rendered tool input/output string
    Since_compact:bool=False, # Only the conversation since the last compaction?
    strip_Tools:bool=False, # Drop tool calls and their results entirely?
    Recent:bool=False, # Convert the newest session for the current directory instead?
):
    "Write a Claude session or Codex thread to an ipynb dialog"
    if (ref is None) != Recent: raise ValueError('pass exactly one of: a session ref, or -r for the newest')
    host,tpath = recent_sess() if Recent else find_sess(ref)
    dlg = path_dlg(host, tpath, ref, mx=mx, since_compact=Since_compact, strip_tools=strip_Tools)
    path = Path(Out) if Out else Path(dlg.name)
    if path.suffix!='.ipynb': path = path.with_name(path.name+'.ipynb')
    write_ipynb(dlg, path)
    if is_cli(sess2nb): print(f'{path}: {len(dlg.messages)} messages from {host} {tpath.stem}')
    else: return path

In [ ]:
with tempfile.TemporaryDirectory() as td:
    out = sess2nb('ab12cd34', f'{td}/sess')
    saved = read_ipynb(out)
test_eq(len(saved.messages), len(sess_dlg('ab12cd34', proj).messages))
test_eq(out.name, 'sess.ipynb')
out.name

With `-r` (`Recent`), no ref is given: the newest transcript for the current directory is chosen, here the Claude fixture made newest above. A ref and `-r` together, or neither, is an error:

In [ ]:
old = Path.cwd()
os.chdir(proj)
with tempfile.TemporaryDirectory() as td: n = len(read_ipynb(sess2nb(Recent=True, Out=f'{td}/r.ipynb')).messages)
os.chdir(old)
test_eq(n, len(sess_dlg('ab12cd34', proj).messages))
with expect_fail(contains='exactly one'): sess2nb()
with expect_fail(contains='exactly one'): sess2nb('ab12cd34', Recent=True)
n

## Cleanup

The Claude fixture has to sit in `~/.claude/projects` to be findable, so remove it again along with both scratch homes.

In [ ]:
shutil.rmtree(sd)
shutil.rmtree(proj)
shutil.rmtree(home)